# Importance Matrix Testing Notebook

This notebook is for manual testing of effect size / importance matrix plots.
It uses functions from `postprocess_functions.py` and `plot_importance_matrix.py`.

The importance matrix computes Cohen's d effect size for each input parameter across quartiles of the target variable.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add parent directory to path for imports
sys.path.insert(0, str(Path().resolve().parent))

from ddstartup.postprocessing.postprocess_functions import (
    load_h5_to_dataframe,
    get_input_parameters,
    scale_target,
    apply_filters,
    find_latest_h5_file
)
from ddstartup.postprocessing.plot_importance_matrix import (
    cohen_d,
    compute_effect_size_matrix,
    plot_effect_size_matrix
)
from ddstartup.utils.tools import PARAM_UNITS

print("✅ Imports successful")

## Configuration

In [ ]:
# Configuration - specify directory or files

# Option 1: Automatic - find latest file in specified directory (DEFAULT)
outputs_dir = Path('../outputs')
files_to_analyze = []

latest_h5_file = find_latest_h5_file(outputs_dir)
if latest_h5_file:
    print(f"📂 Auto-detected folder: {latest_h5_file.parent.name}")
    print(f"📄 Latest file: {latest_h5_file.name}")
    files_to_analyze = [latest_h5_file]

# Option 2: Analyze all files in a specific folder
# outputs_dir = Path('../outputs/20251008_081422_parametric_T_seeded')
# files_to_analyze = sorted(outputs_dir.glob('*.h5'))
# print(f"📂 Using folder: {outputs_dir.name}")
# print(f"📄 Found {len(files_to_analyze)} file(s): {[f.name for f in files_to_analyze]}")

# Target variable to analyze
target = 'Q_fusion'  # Change this to your desired target variable
print(f"\n🎯 Target variable: {target}")

## Load and Prepare Data

In [ ]:
# Load data
if not files_to_analyze:
    raise ValueError("No files to analyze. Please check the configuration.")

df = load_h5_to_dataframe(files_to_analyze)
print(f"📊 Loaded dataframe with shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

# Get input parameters
inputs = get_input_parameters(df)
print(f"\n🔧 Input parameters ({len(inputs)}): {inputs}")

# Check if target exists
if target not in df.columns:
    print(f"❌ Target '{target}' not found in dataframe")
    print(f"Available columns: {list(df.columns)}")
    raise ValueError(f"Target '{target}' not in dataframe")

print(f"\n📈 Target '{target}' range: [{df[target].min():.2e}, {df[target].max():.2e}]")
print(f"📊 Target statistics:")
print(df[target].describe())

## Apply Filters (Optional)

In [ ]:
# Optional: Apply filters
filters = {
    # Example: 'Q_fusion': {'min': 0},  # Only positive Q_fusion
    # Example: 'Ti_0': {'min': 5e3, 'max': 20e3},  # Temperature range
}

if filters:
    df_filtered = apply_filters(df, filters)
    print(f"🔍 Applied filters: {filters}")
    print(f"📊 Filtered dataframe shape: {df_filtered.shape} (was {df.shape})")
    df = df_filtered
else:
    print("No filters applied")

## Test 1: Compute Cohen's d for Sample Comparison

In [ ]:
# Test the cohen_d function with two sample groups
if len(inputs) > 0:
    test_param = inputs[0]
    
    # Split data into two groups based on target median
    median_target = df[target].median()
    group_low = df[df[target] < median_target][test_param]
    group_high = df[df[target] >= median_target][test_param]
    
    effect_size = cohen_d(group_high, group_low)
    
    print(f"Cohen's d calculation for '{test_param}':")
    print(f"  Group LOW (target < median):  mean={group_low.mean():.3e}, std={group_low.std():.3e}, n={len(group_low)}")
    print(f"  Group HIGH (target >= median): mean={group_high.mean():.3e}, std={group_high.std():.3e}, n={len(group_high)}")
    print(f"  Cohen's d = {effect_size:.3f}")
    print(f"\n  Interpretation:")
    print(f"    |d| < 0.2  : negligible effect")
    print(f"    |d| < 0.5  : small effect")
    print(f"    |d| < 0.8  : medium effect")
    print(f"    |d| >= 0.8 : large effect")
else:
    print("❌ No input parameters available for testing")

## Test 2: Compute Effect Size Matrix (4 Quartiles)

In [ ]:
# Compute the full effect size matrix
if len(inputs) > 0:
    print(f"Computing effect size matrix for {len(inputs)} inputs and 4 quartiles...\n")
    
    effects = compute_effect_size_matrix(df, target, inputs, quartiles=4)
    
    print("Effect Size Matrix (Cohen's d):")
    print("="*60)
    print(effects)
    print("="*60)
    
    # Identify most important parameters per quartile
    print("\nMost important parameters per quartile (by |Cohen's d|):")
    for col in effects.columns:
        abs_effects = effects[col].abs()
        top_param = abs_effects.idxmax()
        top_value = effects.loc[top_param, col]
        print(f"  {col}: {top_param} (d={top_value:.3f})")
else:
    print("❌ No input parameters available for effect size matrix")

## Test 3: Plot Effect Size Matrix Heatmap

In [ ]:
# Create the full importance matrix plot
if len(inputs) > 0:
    # Create output directory for test plots
    test_outputs_dir = Path('../outputs/manual_test_importance')
    test_outputs_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Creating importance matrix heatmap for '{target}'...")
    
    effects_result = plot_effect_size_matrix(
        df, 
        target=target,
        inputs=inputs,
        outputs_dir=test_outputs_dir,
        plot_name=f'importance_matrix_{target}',
        save_csv=True
    )
    
    print(f"\n✅ Plot saved to {test_outputs_dir}")
    print(f"   - PNG: importance_matrix_{target}.png")
    print(f"   - CSV: importance_matrix_{target}_effects.csv")
    
    # Display the saved image
    from IPython.display import Image, display
    display(Image(filename=test_outputs_dir / f'importance_matrix_{target}.png'))
else:
    print("❌ No input parameters available for plotting")

## Test 4: Custom Number of Quartiles

In [ ]:
# Test with different number of quartiles (e.g., 3 or 5)
if len(inputs) > 0:
    n_quartiles = 5
    
    print(f"Computing effect size matrix with {n_quartiles} quartiles...\n")
    
    effects_custom = compute_effect_size_matrix(df, target, inputs, quartiles=n_quartiles)
    
    print(f"Effect Size Matrix ({n_quartiles} quartiles):")
    print("="*70)
    print(effects_custom)
    print("="*70)
    
    # Plot custom quartiles
    plt.figure(figsize=(max(8, len(inputs)*0.5), 7))
    sns.heatmap(effects_custom.astype(float), cmap='vlag', center=0, annot=True, fmt='.2f')
    plt.title(f"Effect Size (Cohen's d) - {n_quartiles} Quartiles — {target}")
    plt.tight_layout()
    plt.savefig(test_outputs_dir / f'importance_matrix_{target}_{n_quartiles}q.png', dpi=150)
    plt.show()
    
    print(f"\n✅ Custom quartile plot saved")
else:
    print("❌ No input parameters available")

## Test 5: Subset of Input Parameters

In [ ]:
# Test with a subset of input parameters
if len(inputs) > 3:
    # Select top 5 inputs by variance or first 5
    selected_inputs = inputs[:5]
    
    print(f"Computing effect size matrix for selected inputs: {selected_inputs}\n")
    
    effects_subset = plot_effect_size_matrix(
        df,
        target=target,
        inputs=selected_inputs,
        outputs_dir=test_outputs_dir,
        plot_name=f'importance_matrix_{target}_subset',
        save_csv=True
    )
    
    # Display the saved image
    from IPython.display import Image, display
    display(Image(filename=test_outputs_dir / f'importance_matrix_{target}_subset.png'))
    
    print(f"\n✅ Subset importance matrix saved")
else:
    print("ℹ️ Skipping subset test (need more than 3 inputs)")

## Test 6: Analyze Multiple Targets

In [ ]:
# Test with multiple target variables
output_columns = [col for col in df.columns if col not in inputs and col != 'index']
targets_to_test = output_columns[:3] if len(output_columns) >= 3 else output_columns

if len(targets_to_test) > 1:
    print(f"Testing importance matrix for multiple targets: {targets_to_test}\n")
    
    for tgt in targets_to_test:
        print(f"\n{'='*60}")
        print(f"Target: {tgt}")
        print(f"{'='*60}")
        
        effects = plot_effect_size_matrix(
            df,
            target=tgt,
            inputs=inputs,
            outputs_dir=test_outputs_dir,
            plot_name=f'importance_matrix_{tgt}',
            save_csv=False  # Don't save CSV for this test
        )
        
        # Show top 3 most important parameters
        mean_abs_effect = effects.abs().mean(axis=1).sort_values(ascending=False)
        print(f"\nTop 3 most important parameters for {tgt}:")
        for i, (param, value) in enumerate(mean_abs_effect.head(3).items(), 1):
            print(f"  {i}. {param}: mean |d| = {value:.3f}")
    
    print(f"\n✅ All importance matrices saved to {test_outputs_dir}")
else:
    print("ℹ️ Not enough output variables for multi-target test")

## Summary

### Importance Matrix Functions Tested:

1. ✅ **cohen_d**
   - Computes Cohen's d effect size between two samples
   - Measures standardized difference in means

2. ✅ **compute_effect_size_matrix**
   - Computes effect size for all inputs across quartiles
   - Returns DataFrame with inputs × quartiles
   - Customizable number of quartiles

3. ✅ **plot_effect_size_matrix**
   - Creates heatmap visualization
   - Saves PNG and CSV files
   - Color-coded by effect magnitude (red=positive, blue=negative)

### Interpretation Guide:

**Cohen's d Effect Size:**
- |d| < 0.2: Negligible effect
- 0.2 ≤ |d| < 0.5: Small effect
- 0.5 ≤ |d| < 0.8: Medium effect
- |d| ≥ 0.8: Large effect

**How to Read the Matrix:**
- Each row = one input parameter
- Each column = one quartile of the target
- Cell value = Cohen's d when comparing:
  - Samples in that quartile vs. all other samples
  - For that specific input parameter

**Key Use Cases:**
- Identify which inputs most influence the target
- Understand parameter importance at different output ranges
- Guide further analysis or optimization efforts

### Output Files:
- PNG: Heatmap visualization
- CSV: Raw effect size values for further analysis